T1:

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import  MinMaxScaler

T2

In [2]:
df=pd.read_csv("/content/comprehensive_mutual_funds_data.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/content/comprehensive_mutual_funds_data.csv'

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
df.nunique()

In [ ]:
df.isnull().sum()

T3:

In [ ]:
percentage=(df.isnull().sum()/len(df))*100
percentage

In [ ]:
df["returns_3yr"] = df["returns_3yr"].fillna(df["returns_3yr"].mean())

In [ ]:
df['returns_5yr']=df['returns_5yr'].fillna(df['returns_5yr'].mean())

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.head()

In [ ]:
numeric_cols=df.select_dtypes(include=np.number).columns.tolist() #selecting the column with datatyes are numeric
print(numeric_cols) # we're doing this inorder to chech if have any other  than (numeric values) corrupt

In [ ]:
for col in numeric_cols:
  df[col]=(df[col].
           astype(str) #string conversion
           .str.replace('%',"",regex=False)
           .str.replace(",","",regex=False) # regex for pattern recognisation
           )
  df[col]=pd.to_numeric(df[col],errors="coerce") # string to numeric and again replacing errors with NAN
  df[col]=df[col].fillna(df[col].median()) # the data is too less therfore we are utilizing medaian



In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
df.fillna(df.median(numeric_only=True), inplace=True) #replacement null values

In [ ]:
df.fillna("Unknown", inplace=True) #replace null values with unknown <string types columns>

EDA STEPS:

In [ ]:
df["category"].value_counts() #chegging unique values in DS

In [ ]:
df['amc_name'].value_counts()

In [ ]:
df['expense_ratio'].min()

In [ ]:
df["returns_5yr"].max()

In [ ]:
df.corr(numeric_only=True) #chegging correlation between numerical values

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), # heat map is utilized to understand co relation
            annot=True,# for number values to be visible
            cmap="coolwarm")
plt.show() #1 is for same column,-ve is bad relation, +ve is good relation

In [ ]:
df.columns

T5:

In [ ]:
features = [
    'expense_ratio',
    'returns_1yr',
    'returns_3yr',
    'returns_5yr',
    'sharpe',
    'sortino',
    'alpha',
    'beta'
]# we're trying calculate to composite score (evry columns numerical values * weights)and range

score_df = df[features].copy() # inorder to let the original data remain disturbed
score_df.head()

In [ ]:
score_df.isnull().sum()

In [ ]:
score_df = score_df.apply(pd.to_numeric, errors='coerce') #coerce coverts errors values into NA

In [ ]:
score_df.dtypes

In [ ]:
scaler=MinMaxScaler() #scalling the numerical values of the columns
normalized=scaler.fit_transform(score_df) #calculate statics values for normalisation

In [ ]:
normalized = pd.DataFrame(
normalized,
columns=features
) #normalizing numerical columns

In [ ]:
score_df

In [ ]:
normalized['expense_ratio'] = 1-normalized['expense_ratio'] #as per the doc

In [ ]:
normalized['beta']=1-normalized['beta']

In [ ]:
normalized['Composite Score'] = (
    normalized['returns_1yr'] * 0.20 +
    normalized['returns_3yr'] * 0.20 +
    normalized['returns_5yr'] * 0.20 +
    normalized['expense_ratio'] * 0.15 +
    normalized['sharpe'] * 0.10 +
    normalized['sortino'] * 0.05 +
    normalized['alpha'] * 0.05 +
    normalized['beta'] * 0.05
) # feature engineering,adding new feature i.e. composite score

In [ ]:
df['Composite Score'] = normalized['Composite Score'] #changing value in the main datset

In [ ]:
df['Rank']=df['Composite Score'].rank(
ascending=False,
method='dense'
) #feature engineering i.e. rank

In [ ]:
df=df.sort_values(
'Composite Score',
ascending=False
) # sorting wrt score

In [ ]:
top10=df.head(10) #evaluating the best

In [ ]:
top20=df.head(20) #evaluating the best

In [ ]:
top30=df.head(30)#evaluating the best

In [ ]:
df.replace('-', np.nan, inplace=True)

In [ ]:
df.fillna(df.median(numeric_only=True), inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
for col in ['sortino', 'alpha', 'sd', 'beta', 'sharpe']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df.fillna(df.median(numeric_only=True), inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df.to_excel(
"MutualFund_Clean.xlsx",
index=False
) # converting to excel